# Requirement 14: No recovery or resumption

> Application restart, execution-environment restart, failed verification, or significant communication failure must invalidate the session. A retry must use a new environment and new secrets.

**Covered by:** Both

**How it is fulfilled:** Suspend and snapshots are disabled, so there is no state to resume. On any failure the orchestrator destroys the sandbox and starts over with a new sandbox and new key.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
info = sandbox.get()
check("suspend is disabled (no memory or disk state to resume)", not info.lifecycle.auto_suspend.enabled)
check("created from a disk, not a snapshot", info.sources_ref.snapshot is None)

PASS  suspend is disabled (no memory or disk state to resume)
PASS  created from a disk, not a snapshot


In [3]:
key1 = install_session_key(sandbox, session_id)
# A verification failure or communication failure: destroy it
cleanup(sandbox)
check("failed session is destroyed", not list(orchestrator().list_sandboxes(labels={"session": session_id})))

PASS  failed session is destroyed


In [4]:
# The retry is a new environment with new secrets
retry, retry_id = new_session()
key2 = install_session_key(retry, retry_id)
check("new sandbox id", retry.sandbox_id != sandbox.sandbox_id)
check("new session id", retry_id != session_id)
check("new key", key1 != key2)
check("the old key does not work in the new session",
      "REJECTED" in send(retry, signed_request(key1, retry_id, 1, [1])))
cleanup(retry)

PASS  new sandbox id
PASS  new session id
PASS  new key


PASS  the old key does not work in the new session


**Caveat:** The platform supports suspend and snapshot, so keeping them off is a configuration the orchestrator verifies (requirement 3). The restart rule itself is the orchestrator's logic.